In [ ]:
# StyleForge Training Notebook
# Neural Style Transfer with Multi-Style CIN Model

!unzip -o project.zip -d .
!pip install -r requirements.txt

import os
os.makedirs("models", exist_ok=True)
print("✓ Environment ready")

In [ ]:
# Verify style images are present
import os
from pathlib import Path

style_files = [
    "styles/starry_night.jpg",
    "styles/great_wave.jpg",
    "styles/girl_pearl.jpg",
    "styles/composition_viii.jpg",
    "styles/water_lilies.jpg",
]

missing = [f for f in style_files if not os.path.exists(f)]
if missing:
    print(f"⚠ Missing style images: {', '.join(missing)}")
    print("Download them from the README or re-clone the repo.")
    raise FileNotFoundError("Style images missing")
else:
    print(f"✓ All {len(style_files)} style images present")

In [ ]:
# Download training data and VGG16 weights
import os
import shutil

MINI_DATASET = False  # default: full COCO train2017 (~19GB, 118K images)
# MINI_DATASET = True  # uncomment for a mini test run: COCO val2017 (5K images, ~800MB)

if MINI_DATASET:
    DATASET_URL = "http://images.cocodataset.org/zips/val2017.zip"
    ZIP_NAME = "val2017.zip"
    DATASET_DIR = "training_content/val2017"
    DATASET_LABEL = "mini COCO val2017 (5K images, ~800MB)"
else:
    DATASET_URL = "http://images.cocodataset.org/zips/train2017.zip"
    ZIP_NAME = "train2017.zip"
    DATASET_DIR = "training_content/train2017"
    DATASET_LABEL = "COCO train2017 (~19GB, 118K images)"

if not os.path.exists(DATASET_DIR):
    print(f"Downloading {DATASET_LABEL}...")
    if os.path.exists("training_content"):
        shutil.rmtree("training_content")

    !wget -q --show-progress {DATASET_URL} -O {ZIP_NAME}
    !mkdir -p training_content
    !unzip -q {ZIP_NAME} -d training_content

    if os.path.exists(ZIP_NAME):
        os.remove(ZIP_NAME)
    print(f"✓ Dataset Ready at: {DATASET_DIR}")
else:
    print("✓ Dataset already exists.")

if not os.path.exists("models/vgg16.pth"):
    print("Downloading VGG16 Weights...")
    !wget -q --show-progress https://download.pytorch.org/models/vgg16-397923af.pth -O models/vgg16.pth
    print("✓ VGG Weights Ready.")
else:
    print("✓ VGG Weights already exist.")


In [ ]:
# Multi-style CIN Training
# Trains a single model that handles all 5 curated styles via conditional instance normalization

STYLE_IMAGES = ",".join([
    "styles/starry_night.jpg",
    "styles/great_wave.jpg",
    "styles/girl_pearl.jpg",
    "styles/composition_viii.jpg",
    "styles/water_lilies.jpg",
])
MODEL_NAME = "multistyle"
NUM_STYLES = 5
# EPOCHS = 1  # uncomment for a quick mini test run
EPOCHS = 8

print(f"Starting CIN Training with {NUM_STYLES} styles...")
print(f"Style images: {STYLE_IMAGES}")

# Tuned for A100 80GB (Colab premium). For a T4 (16GB): --batch-size 32 --lr 3e-3 --num-workers 2.

# Resume from the latest epoch checkpoint if one exists (Colab sessions die)
import glob
import re

ckpts = sorted(glob.glob("models/cin_ckpt_epoch_*.pth"),
               key=lambda p: int(re.search(r"epoch_(\d+)", p).group(1)))
resume_arg = ""
if ckpts:
    last_epoch = int(re.search(r"epoch_(\d+)", ckpts[-1]).group(1))
    if last_epoch < EPOCHS:
        print(f"Resuming from {ckpts[-1]} (epoch {last_epoch + 1})")
        resume_arg = f"--resume {ckpts[-1]}"

!python -u -m styleforge.train_cin cin \
    --dataset training_content \
    --style-images {STYLE_IMAGES} \
    --save-model-dir models \
    --save-model-name {MODEL_NAME} \
    --cuda 1 \
    --amp 1 \
    --epochs {EPOCHS} \
    --batch-size 128 \
    --image-size 256 \
    --style-size 512 \
    --content-weight 1e5 \
    --style-weight 1e9 \
    --lr 4e-3 \
    --val-fraction 0.02 \
    --num-workers 8 \
    --deterministic 0 \
    --checkpoint-model-dir models \
    --checkpoint-interval 2000 {resume_arg}

print("✓ CIN Training Finished!")

In [ ]:
# Download trained model
from google.colab import files
import os

target_file = f"models/{MODEL_NAME}.pth"

if os.path.exists(target_file):
    print(f"Downloading: {target_file}")
    files.download(target_file)
    print(f"\nModel size: {os.path.getsize(target_file) / 1024 / 1024:.1f} MB")
else:
    print("⚠ Model file not found. Training may have failed.")

if os.path.exists('models/cin_loss_plot.png'):
    files.download('models/cin_loss_plot.png')

In [ ]:
# Optional: Test style interpolation locally after downloading the model

import torch
from styleforge.cin import CINTransformer

model = CINTransformer(num_styles=NUM_STYLES)
state_dict = torch.load(target_file, map_location="cpu", weights_only=True)
model.load_state_dict(state_dict)
model.eval()

print(f"✓ Model loaded: {NUM_STYLES} styles")
print(f"✓ Parameters: {sum(p.numel() for p in model.parameters()) / 1024 / 1024:.1f}M")

# Test inference with different styles
dummy_input = torch.randn(1, 3, 256, 256)
for style_id in range(NUM_STYLES):
    with torch.no_grad():
        output = model(dummy_input, torch.tensor([style_id]))
    print(f"Style {style_id}: output shape {output.shape}, range [{output.min():.1f}, {output.max():.1f}]")

# Test interpolation between style 0 and style 1
with torch.no_grad():
    interp_output = model.forward_interpolated(dummy_input, style_id_a=0, style_id_b=1, alpha=0.5)
print(f"\n✓ Interpolation (0.5 blend): output shape {interp_output.shape}")
print("✓ All tests passed!")